In [1]:
import urllib.request; exec(urllib.request.urlopen('https://aic-data.aiffel.io/api/colab/setup?t=3vvc47bz').read().decode())

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit



⏳  터널 준비 확인 중...

✅  터널 생성 완료!
🔗  URL: https://which-remember-writer-related.trycloudflare.com

아래 [URL 복사] 버튼을 누른 뒤 웹앱 연결창에 붙여넣으세요. (이 탭은 열어두세요)


✅ 웹앱에 자동 연결 요청을 보냈습니다. 잠시 후 웹앱 화면이 연결됩니다.


In [2]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 설치 (필요할 때만) + import + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요. polars가 이번 노트북의 새 도구입니다.
# !pip install numpy pandas polars matplotlib seaborn psutil -q

import os
import gc
import time
import platform
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")   # 학습 중 경고 메시지를 잠시 숨깁니다.

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록 운영체제별로 분기)
system = platform.system()
if system == "Darwin":          # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":       # Windows
    plt.rcParams["font.family"] = "Malgun Gothic"
else:                            # Linux 등
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False   # 마이너스 부호 깨짐 방지
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)
print("polars :", pl.__version__)

준비 완료! 라이브러리 버전을 확인합니다.
numpy  : 2.0.2
pandas : 2.2.2
polars : 1.35.2


In [3]:
# ─────────────────────────────────────────────
# 모두마켓 데이터 생성 — 오늘 쓸 데이터를 한 번에 준비합니다.
# (이번에는 일부러 크게 만듭니다. 메모리 한계가 슬슬 보이도록.)
# ─────────────────────────────────────────────
np.random.seed(42)

DATA_DIR = Path("./moodumarket_big")
DATA_DIR.mkdir(exist_ok=True)

# 1) 고객(customers) — 5만 명
n_customers = 50_000
customers = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(6)}" for i in range(1, n_customers + 1)],
    "age": np.clip(np.random.normal(35, 9, n_customers).round(), 14, 80).astype(int),
    "gender": np.random.choice(["M", "F"], n_customers),
    "region": np.random.choice(
        ["서울", "경기", "부산", "인천", "대구", "광주", "대전", "기타"], n_customers
    ),
    "membership": np.random.choice(["basic", "premium", "vip"], n_customers, p=[0.7, 0.25, 0.05]),
})

# 2) 상품(products) — 1천 종
n_products = 1_000
products = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(5)}" for i in range(1, n_products + 1)],
    "category": np.random.choice(["패션", "뷰티", "식품", "가전", "도서"], n_products),
    "price": np.random.choice([9900, 19900, 29900, 49900, 89900, 129900], n_products),
})

# 3) 주문(orders) — 20만 건
n_orders = 200_000
orders = pd.DataFrame({
    "order_id": np.arange(1, n_orders + 1),
    "customer_id": np.random.choice(customers["customer_id"], n_orders),
    "product_id": np.random.choice(products["product_id"], n_orders),
    "quantity": np.random.choice([1, 1, 1, 2, 2, 3], n_orders),
    "amount": np.random.choice([9900, 19900, 29900, 49900, 89900, 129900], n_orders).astype(float),
    "channel": np.random.choice(["web", "app"], n_orders, p=[0.45, 0.55]),
    "order_date": pd.to_datetime("2025-01-01")
                  + pd.to_timedelta(np.random.randint(0, 120, n_orders), unit="D"),
})

# 4) 웹 로그(web_logs) — 50만 건 (오늘의 주인공)
n_logs = 500_000
web_logs = pd.DataFrame({
    "log_id": np.arange(1, n_logs + 1),
    "ts": pd.to_datetime("2025-04-01")
          + pd.to_timedelta(np.random.randint(0, 86400 * 30, n_logs), unit="s"),
    "user_id": np.random.choice(customers["customer_id"], n_logs),
    "session_id": np.random.randint(1, 80_000, n_logs),
    "page": np.random.choice(
        ["home", "list", "detail", "cart", "checkout", "mypage", "search"],
        n_logs,
        p=[0.30, 0.25, 0.20, 0.10, 0.05, 0.05, 0.05],
    ),
    "device": np.random.choice(["mobile", "desktop", "tablet"], n_logs, p=[0.70, 0.25, 0.05]),
    "status_code": np.random.choice([200, 200, 200, 200, 304, 404, 500], n_logs),
    "response_ms": np.clip(np.random.gamma(2.0, 80, n_logs), 5, 5000).round().astype(int),
    "bytes_sent": np.random.randint(500, 200_000, n_logs),
})

# CSV로 저장해두면 청크 처리·Polars 입출력 비교 시 같은 파일을 함께 씁니다.
orders_csv = DATA_DIR / "orders.csv"
logs_csv = DATA_DIR / "web_logs.csv"
orders.to_csv(orders_csv, index=False)
web_logs.to_csv(logs_csv, index=False)

print("모두마켓 데이터 생성 완료")
print(f"  customers : {customers.shape}")
print(f"  products  : {products.shape}")
print(f"  orders    : {orders.shape}  →  {orders_csv} ({orders_csv.stat().st_size/1024/1024:.1f} MB)")
print(f"  web_logs  : {web_logs.shape}  →  {logs_csv} ({logs_csv.stat().st_size/1024/1024:.1f} MB)")

모두마켓 데이터 생성 완료
  customers : (50000, 5)
  products  : (1000, 3)
  orders    : (200000, 7)  →  moodumarket_big/orders.csv (8.9 MB)
  web_logs  : (500000, 9)  →  moodumarket_big/web_logs.csv (32.3 MB)


In [7]:
# 스스로 해보자! ✏️
# 1. orders의 현재 메모리를 측정해서 출력하세요.
# 2. quantity, channel, customer_id, product_id의 dtype을 줄여 'orders_small'을 만드세요.
# 3. 줄인 뒤 메모리와 절감률을 출력하세요.

# 예제: 다이어트 전 — 현재 dtype과 메모리 확인
print("=== orders: Before — 컬럼별 dtype과 메모리 ===")
print(orders.dtypes)
print("메모리 합계:", orders.memory_usage(deep=True).sum() / 1024 / 1024, "MB")

# 다이어트 시작
print("\n=== orders: 다이어트 시작 ===")
orders_small = orders.copy()
orders_small["order_id"] = orders_small["order_id"].astype("int32")
orders_small["quantity"] = orders_small["quantity"].astype("int8")
orders_small["channel"] = orders_small["channel"].astype("category")
orders_small["amount"] = orders_small["amount"].astype("float32")

b = orders.memory_usage(deep=True).sum() / 1024 / 1024
a = orders_small.memory_usage(deep=True).sum() / 1024 / 1024
print(f"Before: {b:.2f} MB")
print(f"After : {a:.2f} MB")
print(f"절감   : {(1 - a / b) * 100:.1f}%")

=== orders: Before — 컬럼별 dtype과 메모리 ===
order_id                int64
customer_id            object
product_id             object
quantity                int64
amount                float64
channel                object
order_date     datetime64[ns]
dtype: object
메모리 합계: 37.193424224853516 MB

=== orders: 다이어트 시작 ===
Before: 37.19 MB
After : 24.61 MB
절감   : 33.8%


In [9]:
# 예제 1: Polars로 CSV 읽기 — pl.read_csv
t0 = time.perf_counter()
logs_pl = pl.read_csv(logs_csv, try_parse_dates=True)
elapsed_pl = time.perf_counter() - t0

# pandas와 같은 작업: 시간 측정용
t0 = time.perf_counter()
logs_pd = pd.read_csv(logs_csv, parse_dates=["ts"])
elapsed_pd = time.perf_counter() - t0

print(f"Polars pl.read_csv : {elapsed_pl:.2f} 초   shape = {logs_pl.shape}")
print(f"pandas pd.read_csv : {elapsed_pd:.2f} 초   shape = {logs_pd.shape}")
print()
print("=== Pandas head ===")
print(logs_pd.head())
print("\n=== Polars head ===")
print(logs_pl.head())

Polars pl.read_csv : 0.23 초   shape = (500000, 9)
pandas pd.read_csv : 1.20 초   shape = (500000, 9)

=== Pandas head ===
   log_id                  ts  user_id  ...  status_code response_ms bytes_sent
0       1 2025-04-21 15:33:35  C044419  ...          200         277      84795
1       2 2025-04-14 07:53:28  C011729  ...          404          67       5475
2       3 2025-04-25 21:46:48  C028864  ...          500         119     104296
3       4 2025-04-21 18:00:48  C000102  ...          304         241      79304
4       5 2025-04-14 10:38:40  C016620  ...          404         261      86505

[5 rows x 9 columns]

=== Polars head ===
shape: (5, 9)
┌────────┬─────────────┬─────────┬────────────┬───┬─────────┬────────────┬────────────┬────────────┐
│ log_id ┆ ts          ┆ user_id ┆ session_id ┆ … ┆ device  ┆ status_cod ┆ response_m ┆ bytes_sent │
│ ---    ┆ ---         ┆ ---     ┆ ---        ┆   ┆ ---     ┆ e          ┆ s          ┆ ---        │
│ i64    ┆ datetime[μs ┆ str     ┆ i64 

In [12]:
# 스스로 해보자! ✏️
# 1. 본인이 보고 싶은 쿼리를 정합니다. (예: 채널이 'mobile'이고 status==200인 행의 page별 평균 bytes_sent)
# 2. 같은 쿼리를 pandas / polars eager / polars lazy 로 작성하세요.
# 3. benchmark()로 세 가지 시간을 출력하세요.

# 여기에 코드를 작성하세요

def benchmark(label, func, n=3):
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        func()
        times.append(time.perf_counter() - t0)
    avg_ms = sum(times) / n * 1000
    print(f"{label:24s}: {avg_ms:7.1f} ms   (n={n}, min={min(times)*1000:.1f}ms)")
    return avg_ms

def q5_pandas():
    df = pd.read_csv(logs_csv)
    return df[df["status_code"] == 200].groupby("page").size()

def q5_polars_eager():
    return (
        pl.read_csv(logs_csv)
        .filter(pl.col("status_code") == 200)
        .group_by("page")
        .agg(pl.len())
    )

def q5_polars_lazy():
    return (
        pl.scan_csv(logs_csv)
        .filter(pl.col("status_code") == 200)
        .group_by("page")
        .agg(pl.len())
        .collect()
    )

benchmark("pandas",        q5_pandas)
benchmark("polars (eager)", q5_polars_eager)
benchmark("polars (lazy)",  q5_polars_lazy)


pandas                  :   804.3 ms   (n=3, min=795.0ms)
polars (eager)          :   194.1 ms   (n=3, min=188.2ms)
polars (lazy)           :    86.3 ms   (n=3, min=84.7ms)


86.2868483333538

# 스스로 해보자! ✏️
다음 시나리오를 읽고, 어떤 도구를 1차로 추천하시겠어요? 그리고 그 이유를 한 줄로 적어보세요.
- 시나리오 A — 5천 행짜리 매출 요약 보고서. 매주 손으로 돌림.
- 시나리오 B — 매일 자동 실행되는 광고 로그(2,000만 행) ETL 파이프라인.
- 시나리오 C — 50만 행짜리 고객 분석. 동료가 모두 pandas만 씀.

# 위 시나리오 A, B, C에 대해 본인의 선택을 주석으로 적어보세요.

- A: Polars 사용. 1천행이 넘어가고 재반복 하므로 자원 낭비를 줄임.
- B: 로그 파일 류의 데이터는 보통 너무 크고 반복 작업이므로 Polars 를 쓰고 다이어트, 청크, lazy 모두 사용해야 함.
- C: 50만 행이고 동료가 Pandas 만 쓴다면 일단 다이어트, 청크 사용해야 하지만,
너무 데이터가 많고 향후 이런 큰 고객 데이터를 또 다루어야 한다면
Polars 를 쓰고 다이어트, 청크, lazy 모드 활용해야 함.

In [21]:
try:
    import psutil
    proc = psutil.Process(os.getpid())
    rss_mb = proc.memory_info().rss / 1024 / 1024
    print(f"📊 현재 노트북 프로세스 메모리 사용량(RSS): {rss_mb:.1f} MB")
    print("→ 위 web_logs 외에도 customers, orders, products 등이 함께 들어있어 더 큽니다.")
except ImportError:
    print("psutil 미설치 — `!pip install psutil` 로 설치하면 노트북 전체 메모리도 볼 수 있어요.")
def rss_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024

# 시나리오 1 — pandas + dtype 최적화 (전체 분석을 한 함수로 묶음)

def analyze_pandas_optimized(csv_path):
    dtype_map = {
        "log_id": "int32", "session_id": "int32",
        "page": "category", "device": "category",
        "status_code": "int16", "response_ms": "int16", "bytes_sent": "int32",
    }
    df = pd.read_csv(csv_path, dtype=dtype_map, parse_dates=["ts"])

    # (1) 페이지별 응답 시간 분포 — 평균·중앙값·표준편차
    by_page = df.groupby("page", observed=True)["response_ms"].agg(
        ["mean", "median", "std"]
    ).round(2)

    # (2) 디바이스별 트래픽 점유율 — 총 bytes_sent의 디바이스별 비율
    dev_total = df.groupby("device", observed=True)["bytes_sent"].sum()
    dev_share = (dev_total / dev_total.sum() * 100).round(1).rename("share_pct")

    # (3) 에러 발생 패턴 — status_code 400+ 의 페이지별 카운트
    err_count = (df[df["status_code"] >= 400]
                 .groupby("page", observed=True).size()
                 .rename("n_errors").sort_values(ascending=False))

    return {"by_page": by_page, "dev_share": dev_share, "err_count": err_count}

gc.collect()
before = rss_mb()
t0 = time.perf_counter()
res_pd = analyze_pandas_optimized(logs_csv)
elapsed_pd_full = time.perf_counter() - t0
peak_pd_full = rss_mb() - before

print(f"\n📊 [pandas + dtype] web_log 분석 소요 시간: {elapsed_pd_full:.2f}초, 메모리 증가량: {peak_pd_full:.1f} MB\n")
print("(1) 페이지별 응답 시간"); print(res_pd["by_page"]); print()
print("(2) 디바이스별 트래픽 점유율(%)"); print(res_pd["dev_share"]); print()
print("(3) 에러 카운트"); print(res_pd["err_count"])

# 시나리오 2 — 청크 처리 (같은 분석을 50K 청크로)
def analyze_chunked(csv_path, chunksize=50_000):
    # 누적 컨테이너
    sum_ms = {}; cnt_ms = {}; ms_lists = {}
    dev_bytes = {}; err_by_page = {}

    for chunk in pd.read_csv(csv_path, chunksize=chunksize,
                              dtype={"page": "category", "device": "category",
                                     "status_code": "int16", "response_ms": "int16"},
                              parse_dates=["ts"]):
        # 페이지별 합계·개수 (평균/표준편차 재계산용으로 모든 값 모음)
        for page, g in chunk.groupby("page", observed=True):
            sum_ms[page] = sum_ms.get(page, 0) + g["response_ms"].sum()
            cnt_ms[page] = cnt_ms.get(page, 0) + g["response_ms"].count()
            ms_lists.setdefault(page, []).append(g["response_ms"].values)

        # 디바이스별 bytes 합
        for dev, g in chunk.groupby("device", observed=True):
            dev_bytes[dev] = dev_bytes.get(dev, 0) + g["bytes_sent"].sum()

        # 에러 페이지별 카운트
        errs = chunk[chunk["status_code"] >= 400].groupby("page", observed=True).size()
        for page, n in errs.items():
            err_by_page[page] = err_by_page.get(page, 0) + n

    # reduce — 페이지별 통계 (mean/median/std)
    by_page = pd.DataFrame({
        page: {
            "mean":   sum_ms[page] / cnt_ms[page],
            "median": float(np.median(np.concatenate(ms_lists[page]))),
            "std":    float(np.concatenate(ms_lists[page]).std()),
        } for page in sum_ms
    }).T.round(2)

    dev_total = pd.Series(dev_bytes)
    dev_share = (dev_total / dev_total.sum() * 100).round(1).rename("share_pct")
    err_count = pd.Series(err_by_page).sort_values(ascending=False).rename("n_errors")

    return {"by_page": by_page, "dev_share": dev_share, "err_count": err_count}

gc.collect()
before = rss_mb()
t0 = time.perf_counter()
res_chunk = analyze_chunked(logs_csv)
elapsed_chunk_full = time.perf_counter() - t0
peak_chunk_full = rss_mb() - before

print(f"\n📊 [chunked] web_log 분석 소요 시간: {elapsed_chunk_full:.2f}초, 메모리 증가량: {peak_chunk_full:.1f} MB")
print("by_page (chunked):"); print(res_chunk["by_page"].sort_index())

# 시나리오 3 — Polars lazy (한 파이프라인)
def analyze_polars_lazy(csv_path):
    lf = pl.scan_csv(csv_path, try_parse_dates=True)

    # (1) 페이지별 응답 시간 — mean/median/std 한 번에
    by_page_lf = lf.group_by("page").agg([
        pl.col("response_ms").mean().alias("mean"),
        pl.col("response_ms").median().alias("median"),
        pl.col("response_ms").std().alias("std"),
    ]).sort("page")

    # (2) 디바이스별 점유율
    dev_lf = lf.group_by("device").agg(
        pl.col("bytes_sent").sum().alias("total_bytes")
    )
    # 점유율은 collect 후에 계산 (전체 합 필요)

    # (3) 에러 페이지별 카운트 — Predicate pushdown 적용 예상
    err_lf = (
        lf.filter(pl.col("status_code") >= 400)
        .group_by("page")
        .agg(pl.len().alias("n_errors"))
        .sort("n_errors", descending=True)
    )

    by_page = by_page_lf.collect().to_pandas().set_index("page").round(2)
    dev_tot = dev_lf.collect().to_pandas().set_index("device")["total_bytes"]
    dev_share = (dev_tot / dev_tot.sum() * 100).round(1).rename("share_pct")
    err_count = err_lf.collect().to_pandas().set_index("page")["n_errors"]

    return {"by_page": by_page, "dev_share": dev_share, "err_count": err_count}

gc.collect()
before = rss_mb()
t0 = time.perf_counter()
res_pl = analyze_polars_lazy(logs_csv)
elapsed_pl_full = time.perf_counter() - t0
peak_pl_full = rss_mb() - before

print(f"\n📊 [Polars lazy] web_log 분석 소요 시간: {elapsed_pl_full:.2f}초, 메모리 증가량: {peak_pl_full:.1f} MB\n")
print("by_page (polars):"); print(res_pl["by_page"].sort_index())



📊 현재 노트북 프로세스 메모리 사용량(RSS): 1517.8 MB
→ 위 web_logs 외에도 customers, orders, products 등이 함께 들어있어 더 큽니다.

📊 [pandas + dtype] web_log 분석 소요 시간: 4.01초, 메모리 증가량: 1.9 MB

(1) 페이지별 응답 시간
            mean  median     std
page                            
cart      160.27   134.0  113.13
checkout  159.94   134.0  112.94
detail    160.03   135.0  113.00
home      159.86   134.0  112.79
list      159.97   134.0  112.89
mypage    158.34   132.0  111.65
search    160.42   134.0  113.82

(2) 디바이스별 트래픽 점유율(%)
device
desktop    25.1
mobile     69.9
tablet      4.9
Name: share_pct, dtype: float64

(3) 에러 카운트
page
home        42898
list        35593
detail      28298
cart        14360
search       7256
mypage       7244
checkout     7108
Name: n_errors, dtype: int64

📊 [chunked] web_log 분석 소요 시간: 2.36초, 메모리 증가량: 0.0 MB
by_page (chunked):
            mean  median     std
cart      160.27   134.0  113.13
checkout  159.94   134.0  112.94
detail    160.03   135.0  113.00
home      159.86   134.0  112.79
list  

# 📊 웹 로그 처리 성능 분석 보고서

## 🖥️ 메모리 사용 현황

| 항목 | 값 |
|---|---|
| 현재 노트북 프로세스 메모리 사용량 (RSS) | **1,517.8 MB** |

> ⚠️ 위 `web_logs` 외에도 `customers`, `orders`, `products` 등이 함께 로드되어 있어 실제 메모리 사용량은 더 큽니다.

---

## 📈 데이터 분석 결과

### (1) 페이지별 응답 시간 (ms)

| Page | Mean | Median | Std |
|---|---|---|---|
| cart | 160.27 | 134.0 | 113.13 |
| checkout | 159.94 | 134.0 | 112.94 |
| detail | 160.03 | 135.0 | 113.00 |
| home | 159.86 | 134.0 | 112.79 |
| list | 159.97 | 134.0 | 112.89 |
| mypage | 158.34 | 132.0 | 111.65 |
| search | 160.42 | 134.0 | 113.82 |

### (2) 디바이스별 트래픽 점유율

| Device | Share (%) |
|---|---|
| 📱 Mobile | 69.9% |
| 🖥️ Desktop | 25.1% |
| 📟 Tablet | 4.9% |

### (3) 페이지별 에러 카운트

| Page | Error Count |
|---|---|
| home | 42,898 |
| list | 35,593 |
| detail | 28,298 |
| cart | 14,360 |
| search | 7,256 |
| mypage | 7,244 |
| checkout | 7,108 |

---

## ⚙️ 처리 방식별 성능 비교

| 처리 방식 | 소요 시간 | 메모리 증가량 |
|---|---|---|
| Pandas + dtype 최적화 | 4.01초 | 1.9 MB |
| Chunked 처리 | 2.36초 | 0.0 MB |
| **Polars (Lazy)** | **0.29초** ⚡ | **0.0 MB** |

---

## 🎯 도구 선택 정당화

- **속도**: Polars가 Pandas 대비 **최대 약 14배**, Chunked 방식 대비 **약 8배** 빠름. 데이터 규모가 커질수록 격차는 더 벌어짐.
- **메모리 효율**: Chunked 처리부터 메모리 피크가 눈에 띄게 줄어들며, Polars는 증가량이 사실상 0에 수렴 → 자원 절약 효과 큼.
- **팀 차원 제언**: 현재 팀이 Pandas 중심으로 작업하고 있더라도, 회사 자원의 효율적 활용을 위해 팀원들이 최소한 점진적으로라도 **Polars 환경 구축**을 검토하도록 권장할 필요가 있음.